# 第 2 章实践：双缓冲向量算子

本章实践围绕一个主题：**把单缓冲算子改造成双缓冲**，综合运用参数标注、Tile 规划、轮转与同步知识。


## 环境准备

本节代码面向 Ascend 950 环境。运行前先完成基础环境配置（详见 01.06）：设置设备号、关闭设备后端自动加载，并完成导入自检。

In [ ]:
import os

os.environ.setdefault('TILE_FWK_DEVICE_ID', '0')
os.environ['TORCH_DEVICE_BACKEND_AUTOLOAD'] = '0'

import pypto_pro.language as pl
import torch
import torch_npu

print('导入完成；设备编译、执行与精度仍需在 950 验证')

## 1. 客观题

1. （填空）TileGroup 配置 `mutex_ids=[3, 7]` 的含义是：该组包含 ______ 块 Tile，构成 ______ 缓冲，两块 Tile 的 mutex ID 分别为 ______。
2. （选择）`group[2]` 与 `group.next()` 的区别是：
   A. 没有区别<br/>
   B. 下标访问不推进轮转位置，next() 推进<br/>
   C. 下标访问推进轮转位置，next() 不推进
3. （判断）`make_tensor` 改变 shape/stride 视图时会在 GM 中搬移或复制数据。（对/错）
4. （选择）Vector 通过 MTE3 把中间结果写入 L1（Mat），Cube 再搬入 L0 使用。wait 侧的 pipe 应为：
   A. FIX<br/>B. MTE2<br/>C. MTE1
5. （填空）UB 上 Tile 起始地址的对齐要求是 ______ 字节；L0A/L0B 是 ______ 字节。

In [ ]:
!cat ./answer/02.08_objective_answer.txt

## 2. 编程实践一（简单）：方向标注版向量加法

**任务**：把第 1 章向量加法 Kernel 的 Tensor 参数补上 `pl.Input` / `pl.Output` 方向标注，并用 Profiling 语境解释方向标注的作用（不改变读写语义，只影响上报）。


In [ ]:
# 在此完成实践一
pass

## 3. 编程实践二（中等）：单缓冲 → 双缓冲搬运计算

**任务**：实现一个"搬运 + 计算重叠"的双缓冲算子：输入 [8, 256] FP32，输出 = x * 2.0。要求：

1. 用 `make_tile_group` 建 [4, 256] 的双缓冲（mutex_ids=[0,1]）；
2. 循环中 `next()` 轮转取 Tile，`load` 第 i 块后**立即发起**第 i+1 块的 load，再计算第 i 块并 store——让两次搬运与一次计算在时序上可重叠；
3. 计算用 `pl.mul(tile, tile, 2.0)`；
4. 与 `x * 2.0` 的 PyTorch 结果校验。


In [ ]:
# 在此完成实践二
pass

**提示**：auto_mutex=True 会根据 mutex 元数据自动插入同步；你的循环写法决定了"下一块搬入"与"本块计算"能否重叠。


## 4. 编程实践三（困难）：双输入双缓冲融合

**任务**：实现 `out = x + y`（[8, 256] FP32），但 x 与 y 各自使用独立的**双缓冲**组，且输出用单缓冲。要求：

1. 三个 TileGroup 的 mutex ID 互不冲突（思考怎么分配）；
2. 每轮迭代从两个输入组各 `next()` 一块，load 后计算并 store；
3. 回答：为什么输出可以只用单缓冲？什么情况下输出也需要双缓冲？


In [ ]:
# 在此完成实践三
pass

完成后对照参考答案，重点比较三处：mutex ID 分配、轮转调用位置、以及"搬运发起时机"的写法。


In [ ]:
!cat ./answer/02.08_practice_answer.txt

### 完整参考实现与验收

三个题目的独立参考分别为 `add_direction`、`scale_prefetch`、`add_double`。输入包含带位置编号的递增矩阵，用于发现第二个块重复读取第一块或输出覆盖；另加随机输入检查一般情形。方向标注只改变上报信息。

[参考源码](./src/buffer_examples.py) 可在 950 上从章节目录执行 `python src/buffer_examples.py`。下列单元仅展示源码；设备编译、精度与性能均以实际执行为准。


In [ ]:
!cat ./src/buffer_examples.py


## 5. 本节小结

本章把"第 1 章能跑"升级为"理解每一行为什么这样写"：参数体系、数据对象、内存规划、多缓冲与同步、编译签名。**双缓冲 + auto_mutex** 是第 5 章融合算子流水设计的直接基础。

下一章进入矢量算子开发：Reg 级计算、Tile 级计算与两者的选型。